# Solveit Review-App Provisioning (`fastcdp`)

> Solveit review-app provisioning, bundle packaging, inspection, and teardown via `fastcdp`.

This module defines:
- `ReviewAppInfo` and the `ReviewAppProvisioner` abstract contract.
- Pure helpers (`build_agent_bootstrap_script`, `build_starter_notebook`, `build_review_bundle_zip`) that pack `review_manifest.json`, `inspect_tickets.py`, `review_tickets.ipynb`, and compiled CRAFT `dist/` assets into a single archive.
- `MemoryReviewAppProvisioner` for deterministic offline testing.
- `FastCDPReviewAppProvisioner`, which drives the verified `solve.it.com/dashboard` HTMX endpoints (`/create_instance`, `/power`, `/get_app`, `/delete_instance`) and uploads the bundle via `/upload_file_?folder=.` (`extract_archives=on`), verifying extraction via `/toggle_files_` and `/file_list_?at=.`.

In [ ]:
#| default_exp solveit_cdp

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import asyncio, base64, io, json, tempfile, time, zipfile
from dataclasses import asdict, dataclass, field
from typing import Any
from urllib.parse import urljoin, urlparse
from fastcore.nbio import mk_cell, nb2str, new_nb
from fastcore.utils import *

from craft_support.models import ReviewManifest

try: from craft_dev.build import CRAFT_ROOT_ASSETS, CRAFT_SUB_ASSETS
except ImportError:
    CRAFT_ROOT_ASSETS = ("CRAFT.css", "CRAFT.js")
    CRAFT_SUB_ASSETS = ("craft_md.js", "hotkey_manager.js", "hotkey_builtins.js", "craft_fonts.js", "craft_history.js", "craft_tts.js", "craft_bug_capture.js")

## `ReviewAppInfo` & `ReviewAppProvisioner` Contract

In [ ]:
#| export
@dataclass
class ReviewAppInfo:
    "State and URLs for a provisioned Solveit review app."
    review_app_id: str
    url: str
    branch: str
    status: str = "running"
    manifest_path: str | None = None
    metadata: dict[str, Any] = field(default_factory=dict)

class ReviewAppProvisioner:
    "Abstract contract for provisioning, inspecting, and tearing down Solveit review apps."

    async def provision(
        self,
        *,
        branch: str, # Git branch backing the review app
        manifest: ReviewManifest, # Ticket and PR context bundle
        instance_name: str | None = None, # Optional explicit Solveit instance name
        assets_dir: Path | None = None, # Optional compiled CRAFT `dist/` directory
    ) -> ReviewAppInfo:
        "Create or reuse an isolated Solveit review app attached to `branch`."
        raise NotImplementedError

    async def inspect(self, review_app_id: str) -> ReviewAppInfo | None:
        "Inspect a Solveit review app's current state, or None if not found."
        raise NotImplementedError

    async def teardown(self, review_app_id: str) -> bool:
        "Stop and delete a Solveit review app. Returns True if removed, False if already absent."
        raise NotImplementedError

In [ ]:
show_doc(ReviewAppProvisioner)

## Review Bundle Zip Builder

Each review app receives a zip archive extracted into `/app/data/` containing:
- `review_manifest.json` — assigned tickets, PR info, and Drive file IDs.
- `inspect_tickets.py` — helper script that downloads `dom.html` and `screenshot.png` via `solvedrive`.
- `review_tickets.ipynb` — starter Solveit/Jupyter notebook pre-wired to inspect the assigned tickets.
- Optional CRAFT runtime assets (`CRAFT.css`, `CRAFT.js`, `.craft-assets/*`, `.craft-assets/vscode-icons/*`) when `assets_dir` is provided.

In [ ]:
#| export
def _slugify(name: str, max_len: int = 28) -> str:
    slug = re.sub(r"[^a-z0-9-]+", "-", name.lower()).strip("-")
    return (slug or "review")[:max_len].strip("-")

_BOOTSTRAP_SCRIPT = r"""\"\"\"Auto-generated solveitcrafts review-app ticket helper.

Run inside the Solveit review app to inspect the assigned tickets, sanitized DOM
snapshots, and screenshots directly from Google Drive via `solvedrive`.
\"\"\"
import json
from pathlib import Path
from fastgws import Drive as FGWSDrive
from fastgws.auth import oauth_creds
from solvedrive.core import GDrive, File

MANIFEST = json.loads(Path(__file__).with_name("review_manifest.json").read_text(encoding="utf-8"))

async def load_gdrive(account=None):
    creds = await oauth_creds(account=account)
    return GDrive(FGWSDrive(creds=creds)) if hasattr(GDrive, "init") else GDrive(creds)

async def fetch_ticket_artifacts(ticket_id: str, account=None):
    \"\"\"Download ticket.json, dom.html, and screenshot.png (if present) via solvedrive.\"\"\"
    gd = await load_gdrive(account=account)
    entry = next((t for t in MANIFEST["tickets"] if t["id"] == ticket_id), None)
    if not entry: raise KeyError(f"Ticket {ticket_id} not in review manifest")
    drive = entry.get("drive") or {}
    out = {"ticket": entry, "dom_html": None, "screenshot_bytes": None}
    if drive.get("dom_file_id"):
        f = await File.fetch(gd, drive["dom_file_id"])
        out["dom_html"] = (await f.download()).decode("utf-8", errors="replace")
    if drive.get("screenshot_file_id"):
        f = await File.fetch(gd, drive["screenshot_file_id"])
        out["screenshot_bytes"] = await f.download()
    return out
""".replace(r'\"\"\"', '"""')

def build_agent_bootstrap_script(manifest: ReviewManifest | None = None) -> str:
    "Generate a Python helper placed in the review app so an agent can inspect tickets via solvedrive."
    return _BOOTSTRAP_SCRIPT

def build_starter_notebook(manifest: ReviewManifest) -> str:
    "Build a minimal Jupyter/Solveit notebook pre-wired to inspect the assigned tickets."
    ticket_lines = [f"- **{t['id']}** — {t.get('title', '')}" for t in manifest.tickets]
    md_source = "\n".join([
        f"# Review App: `{manifest.branch}` ({manifest.group_id})", "",
        f"- **PR:** {manifest.pr_url or '(pending)'}",
        f"- **Base branch:** `{manifest.base_branch}`", "",
        "## Assigned Tickets", *(ticket_lines or ["- (none)"]),
    ])
    code_source = (
        "from inspect_tickets import MANIFEST, fetch_ticket_artifacts\n"
        "print('Group:', MANIFEST['group_id'], 'Branch:', MANIFEST['branch'])\n"
        "for t in MANIFEST['tickets']:\n"
        "    print(t['id'], '-', t['title'], '| Drive:', (t.get('drive') or {}).get('folder_url'))\n")
    nb = new_nb([mk_cell(md_source, cell_type='markdown'), mk_cell(code_source, cell_type='code')],
        meta=dict(kernelspec=dict(display_name='Python 3', language='python', name='python3'), language_info=dict(name='python')))
    return nb2str(nb)

def build_review_bundle_zip(
    manifest: ReviewManifest, # Review manifest embedded in the zip
    *,
    assets_dir: Path | None = None, # Optional compiled CRAFT `dist/` directory
) -> tuple[bytes, int]:
    "Pack CRAFT assets, `review_manifest.json`, `inspect_tickets.py`, and starter dialog into a zip."
    buf, count = io.BytesIO(), 0
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
        def _add_file(src: Path, arcname: str) -> None:
            nonlocal count
            if src.is_file():
                zf.write(src, arcname)
                count += 1

        zf.writestr("review_manifest.json", manifest.to_json())
        zf.writestr("inspect_tickets.py", build_agent_bootstrap_script(manifest))
        zf.writestr("review_tickets.ipynb", build_starter_notebook(manifest))
        count += 3

        if assets_dir is not None and assets_dir.is_dir():
            for name in CRAFT_ROOT_ASSETS: _add_file(assets_dir/name, name)
            for name in CRAFT_SUB_ASSETS: _add_file(assets_dir/name, f".craft-assets/{name}")
            icons_dir = assets_dir/"vscode-icons"
            if icons_dir.is_dir():
                for p in sorted(icons_dir.rglob("*")): _add_file(p, f".craft-assets/vscode-icons/{p.relative_to(icons_dir)}")
    return buf.getvalue(), count

In [ ]:
show_doc(build_review_bundle_zip)

In [ ]:
from fastcore.test import test_eq
from craft_support.models import Ticket

In [ ]:
with tempfile.TemporaryDirectory() as td:
    dist = Path(td)/"dist"
    (dist/"CRAFT.js").mk_write("// craft js\n", encoding="utf-8")
    (dist/"CRAFT.css").mk_write("/* craft css */\n", encoding="utf-8")
    (dist/"craft_md.js").mk_write("// md\n", encoding="utf-8")
    (dist/"vscode-icons"/"py.svg").mk_write("<svg/>", encoding="utf-8")

    t = Ticket(id="TKT-20261123-ZIP001", title="Zip test", description="Check bundle")
    manifest = ReviewManifest.from_tickets([t], group_id="GRP-1", branch="support/zip", base_branch="main")
    zbytes, count = build_review_bundle_zip(manifest, assets_dir=dist)
    test_eq(count, 7)
    with zipfile.ZipFile(io.BytesIO(zbytes)) as zf:
        names = set(zf.namelist())
        nb_json = loads(zf.read("review_tickets.ipynb").decode("utf-8"))
    assert {
        "review_manifest.json", "inspect_tickets.py", "review_tickets.ipynb",
        "CRAFT.js", "CRAFT.css", ".craft-assets/craft_md.js", ".craft-assets/vscode-icons/py.svg",
    }.issubset(names)
    test_eq((nb_json["nbformat"], len(nb_json["cells"])), (4, 2))

## `MemoryReviewAppProvisioner`

In [ ]:
#| export
class MemoryReviewAppProvisioner(ReviewAppProvisioner, BasicRepr):
    "In-memory review app provisioner for deterministic unit & integration testing."
    def __init__(self, base_url: str = "https://solve.it.com") -> None:
        self.base_url = base_url.rstrip("/")
        self.instances = {}
        self.manifests = {}
        self.provision_calls, self.teardown_calls = 0, 0

    async def provision(
        self, *, branch: str, manifest: ReviewManifest, instance_name: str | None = None, assets_dir: Path | None = None,
    ) -> ReviewAppInfo:
        self.provision_calls += 1
        app_id = instance_name or f"review-{_slugify(branch)}"
        if app_id in self.instances:
            self.manifests[app_id] = manifest
            return self.instances[app_id]
        info = ReviewAppInfo(review_app_id=app_id, url=f'{self.base_url}/instances/{app_id}', branch=branch, status='running',
            manifest_path=f'/app/data/.craft-review/{app_id}/review_manifest.json',
            metadata=dict(ticket_ids=[t['id'] for t in manifest.tickets]))
        self.instances[app_id] = info
        self.manifests[app_id] = manifest
        return info

    async def inspect(self, review_app_id: str) -> ReviewAppInfo | None: return self.instances.get(review_app_id)

    async def teardown(self, review_app_id: str) -> bool:
        self.teardown_calls += 1
        existed = review_app_id in self.instances
        self.instances.pop(review_app_id, None)
        self.manifests.pop(review_app_id, None)
        return existed

In [ ]:
async def _test_memory_provisioner():
    prov = MemoryReviewAppProvisioner()
    t = Ticket(id="TKT-1", title="Test", description="Desc")
    m = ReviewManifest.from_tickets([t], group_id="GRP-1", branch="support/fix-1", base_branch="main")
    info1 = await prov.provision(branch="support/fix-1", manifest=m)
    info2 = await prov.provision(branch="support/fix-1", manifest=m)
    test_eq(info1.review_app_id, info2.review_app_id)
    assert (await prov.inspect(info1.review_app_id)) is not None
    test_eq(await prov.teardown(info1.review_app_id), True)
    test_eq(await prov.teardown(info1.review_app_id), False)
    assert (await prov.inspect(info1.review_app_id)) is None

await _test_memory_provisioner()

## `FastCDPReviewAppProvisioner`

Drives `solve.it.com/dashboard` using `fastcdp.CDP` with the cookie session saved by `craft-cdp auth` (`.auth/solveit-state.json`):
- Lists instance cards (`[id^="instance-"]`) and checks running state via `a[href*="get_app"]`.
- Creates missing instances with `POST /create_instance` (`instance_name`).
- Powers instances on/off with `POST /power` (`on=1` / `on=0`).
- Follows `/get_app?inst_id=...` to the live instance origin and uploads `review_bundle.zip` to `POST /upload_file_?folder=.` with `extract_archives=on`, verifying presence via `/toggle_files_` and `/file_list_?at=.`.
- Tears down instances cleanly via `POST /delete_instance?inst_id=...` (`name=<inst_name>`) and verifies removal from the dashboard.

In [ ]:
#| export
_LIST_INSTANCES_JS = r"""(() => {
    return [...document.querySelectorAll('[id^="instance-"]')].map(el => {
        const instId = el.id.replace(/^instance-/, '');
        const lines = (el.innerText || '').trim().split('\n').map(s => s.trim()).filter(Boolean);
        const openLink = el.querySelector('a[href*="get_app"]');
        const powerBtn = el.querySelector('[hx-post^="/power"]');
        const editBtn = el.querySelector('[hx-get^="/edit_settings"]');
        return {
            inst_id: instId,
            name: lines[0] || '',
            running: Boolean(openLink),
            open_href: openLink ? openLink.getAttribute('href') : null,
            power_post: powerBtn ? powerBtn.getAttribute('hx-post') : null,
            edit_get: editBtn ? editBtn.getAttribute('hx-get') : null,
        };
    });
})()"""

def _load_cdp_cookies(auth_file: Path) -> list[dict[str, Any]]:
    try:
        from craft_dev.auth import cookies_for_cdp
        return cookies_for_cdp(auth_file)
    except ImportError:
        raw = Path(auth_file).read_json()
        items = raw["cookies"] if isinstance(raw, dict) else raw
        out = []
        for c in items or []:
            if not (c.get("name") and c.get("value")): continue
            d = {k: c[k] for k in ("name", "value", "domain", "path", "httpOnly", "secure", "sameSite") if c.get(k) is not None}
            if (exp := c.get("expires", -1)) and float(exp) > 0: d["expires"] = float(exp)
            out.append(d)
        return out

class FastCDPReviewAppProvisioner(ReviewAppProvisioner, BasicRepr):
    "Provision, inspect, and tear down isolated Solveit review apps via `fastcdp`."
    def __init__(
        self,
        *,
        dashboard_url: str = "https://solve.it.com/dashboard", # Solveit dashboard URL
        auth_file: Path | str | None = None, # Path to `.auth/solveit-state.json`
        headless: bool = True, # Run Chromium headlessly
        state_file: Path | str | None = None, # Local review-app state JSON path
        ready_timeout: float = 120.0, # Seconds to wait for instance boot
    ) -> None:
        self.dashboard_url = dashboard_url
        self.auth_file = Path(auth_file) if auth_file else Path.cwd()/".auth"/"solveit-state.json"
        self.headless = headless
        self.state_file = Path(state_file) if state_file else Path.cwd()/".auth"/"review-apps.json"
        self.ready_timeout = ready_timeout

    @staticmethod
    def _match_instance(instances: list[dict], app_id: str, inst_id: str | None = None) -> dict[str, Any] | None:
        if inst_id and (hit := first(i for i in instances if str(i.get("inst_id")) == str(inst_id))): return hit
        return first(i for i in instances if (i.get("name") or "").strip() == app_id)

### Local State & CDP Browser Session Lifecycle

In [ ]:
#| export
@patch
def _load_state(self: FastCDPReviewAppProvisioner) -> dict[str, dict[str, Any]]:
    try: return self.state_file.read_json() if self.state_file.is_file() else {}
    except Exception: return {}

@patch
def _save_state(self: FastCDPReviewAppProvisioner, state: dict[str, dict]) -> None: self.state_file.write_json(state, indent=2)

@patch
async def _open_session(self: FastCDPReviewAppProvisioner):
    "Launch a `fastcdp` browser, inject Solveit session cookies, and return `(td, cdp, page)`."
    from fastcdp import CDP
    if not self.auth_file.is_file():
        raise RuntimeError(f"Solveit auth state not found at {self.auth_file}. Run `uv run craft-cdp auth` first.")
    cookies = _load_cdp_cookies(self.auth_file)
    td = tempfile.TemporaryDirectory(prefix="craft-support-cdp-")
    cdp = await CDP.launch(user_data_dir=td.name, headless=self.headless, reuse=False)
    try:
        if cookies: await cdp.Storage.setCookies(cookies=cookies)
        return td, cdp, await cdp.new_page()
    except BaseException:
        await cdp.quit()
        td.cleanup()
        raise

@patch
async def _close_session(self: FastCDPReviewAppProvisioner, td: tempfile.TemporaryDirectory, cdp: Any) -> None:
    try: await cdp.quit()
    finally: td.cleanup()

@patch
async def _goto_dashboard(self: FastCDPReviewAppProvisioner, page: Any) -> list[dict[str, Any]]:
    await page.goto(self.dashboard_url, wait="load", timeout=30)
    await asyncio.sleep(0.8)
    cur = await page.eval("location.href") or ""
    if any(tok in urlparse(cur).path.lower() for tok in ("/login", "/signin", "/oauth")):
        raise PermissionError(f"Solveit session expired (redirected to {cur}). Run `uv run craft-cdp auth` to refresh.")
    return await page.eval(_LIST_INSTANCES_JS) or []

### Instance Boot Polling & Bundle Upload

In [ ]:
#| export
@patch
async def _wait_for_instance_running(
    self: FastCDPReviewAppProvisioner, page: Any, app_id: str, inst_id: str | None = None,
) -> dict[str, Any]:
    deadline, powered_on = time.monotonic() + self.ready_timeout, False
    while time.monotonic() < deadline:
        card = self._match_instance(await self._goto_dashboard(page), app_id, inst_id=inst_id)
        if card is not None:
            inst_id = card["inst_id"]
            if card.get("running") and card.get("open_href"): return card
            power_post = card.get("power_post") or ""
            if not powered_on and "on=1" in power_post:
                await page.eval(f"fetch({json.dumps(power_post)}, {{method: 'POST', credentials: 'same-origin'}}).then(r => r.status)")
                powered_on = True
        await asyncio.sleep(2.5)
    raise TimeoutError(f"Timed out after {self.ready_timeout:.0f}s waiting for Solveit instance {app_id!r} to start.")

@patch
async def _upload_bundle_to_instance(
    self: FastCDPReviewAppProvisioner, page: Any, instance_url: str, zip_bytes: bytes,
) -> dict[str, Any]:
    "Upload `review_bundle.zip` into `/app/data/` via Solveit's `/upload_file_?folder=.` endpoint."
    b64 = base64.b64encode(zip_bytes).decode("ascii")
    res = await page.eval(f"""(async () => {{
        const bin = atob({json.dumps(b64)});
        const bytes = new Uint8Array(bin.length);
        for (let i = 0; i < bin.length; i++) bytes[i] = bin.charCodeAt(i);
        const fd = new FormData();
        fd.append('files', new File([bytes], 'review_bundle.zip', {{type: 'application/zip'}}));
        fd.append('extract_archives', 'on');
        const r = await fetch('/upload_file_?folder=.', {{method: 'POST', body: fd, credentials: 'same-origin'}});
        const text = await r.text();
        const toggle = document.getElementById('toggle_files');
        if (toggle && !toggle.checked) {{
            await fetch('/toggle_files_', {{method: 'POST', credentials: 'same-origin'}});
        }}
        const listResp = await fetch('/file_list_?at=.', {{credentials: 'same-origin'}});
        const listHtml = await listResp.text();
        return {{
            status: r.status, ok: r.ok,
            has_manifest: listHtml.includes('review_manifest.json'),
            has_helper: listHtml.includes('inspect_tickets.py'),
            has_notebook: listHtml.includes('review_tickets.ipynb'),
            has_craft_js: listHtml.includes('CRAFT.js'),
            snippet: text.slice(0, 200),
        }};
    }})()""")
    if not res or not res.get("ok"): raise RuntimeError(f"Failed to upload review bundle to {instance_url}: {res}")
    return res

### Provisioning, Inspection & Teardown

In [ ]:
#| export
@patch
async def provision(
    self: FastCDPReviewAppProvisioner, *, branch: str, manifest: ReviewManifest,
    instance_name: str | None = None, assets_dir: Path | None = None,
) -> ReviewAppInfo:
    app_id = instance_name or f"review-{_slugify(branch)}"
    state = self._load_state()
    saved_inst_id = nested_idx(state.get(app_id) or {}, "metadata", "inst_id")

    td, cdp, page = await self._open_session()
    try:
        card = self._match_instance(await self._goto_dashboard(page), app_id, inst_id=saved_inst_id)
        if card is None:
            status = await page.eval(f"""(async () => {{
                const fd = new FormData();
                fd.append('instance_name', {json.dumps(app_id)});
                const r = await fetch('/create_instance', {{method: 'POST', body: fd, credentials: 'same-origin'}});
                return r.status;
            }})()""")
            if not status or int(status) >= 400: raise RuntimeError(f"Failed to create Solveit instance {app_id!r} (HTTP {status}).")

        card = await self._wait_for_instance_running(page, app_id, inst_id=card["inst_id"] if card else None)
        inst_id, open_url = str(card["inst_id"]), urljoin(self.dashboard_url, card["open_href"])
        await page.goto(open_url, wait="load", timeout=45)
        await asyncio.sleep(2.0)
        instance_url = (await page.eval("location.origin + '/'") or open_url).rstrip("/") + "/"

        zip_bytes, file_count = build_review_bundle_zip(manifest, assets_dir=assets_dir)
        upload_res = await self._upload_bundle_to_instance(page, instance_url, zip_bytes)

        bundle_dir = self.state_file.parent/"review-bundles"/app_id
        manifest_file = bundle_dir/"review_manifest.json"
        manifest_file.mk_write(manifest.to_json(), encoding="utf-8")
        (bundle_dir/"inspect_tickets.py").write_text(build_agent_bootstrap_script(manifest), encoding="utf-8")
        (bundle_dir/"review_tickets.ipynb").write_text(build_starter_notebook(manifest), encoding="utf-8")

        info = ReviewAppInfo(
            review_app_id=app_id, url=instance_url, branch=branch, status="running",
            manifest_path="/app/data/review_manifest.json",
            metadata=dict(
                inst_id=inst_id, ticket_ids=[t["id"] for t in manifest.tickets],
                assets_dir=str(assets_dir) if assets_dir else None, uploaded_files=file_count,
                has_manifest=bool(upload_res.get("has_manifest")), has_helper=bool(upload_res.get("has_helper")),
                has_notebook=bool(upload_res.get("has_notebook")), has_craft_js=bool(upload_res.get("has_craft_js")),
                local_manifest_path=str(manifest_file)))
        state[app_id] = asdict(info)
        self._save_state(state)
        return info
    finally: await self._close_session(td, cdp)

@patch
async def inspect(self: FastCDPReviewAppProvisioner, review_app_id: str) -> ReviewAppInfo | None:
    saved = self._load_state().get(review_app_id)
    return ReviewAppInfo(**saved) if saved else None

@patch
async def teardown(self: FastCDPReviewAppProvisioner, review_app_id: str) -> bool:
    state = self._load_state()
    saved = state.get(review_app_id)
    saved_inst_id = nested_idx(saved or {}, "metadata", "inst_id")
    existed_in_state, removed_live = saved is not None, False

    if self.auth_file.is_file():
        td, cdp, page = await self._open_session()
        try:
            card = self._match_instance(await self._goto_dashboard(page), review_app_id, inst_id=saved_inst_id)
            if card is not None:
                inst_id, inst_name = str(card["inst_id"]), str(card.get("name") or review_app_id)
                power_post = card.get("power_post") or ""
                if card.get("running") and "on=0" in power_post:
                    await page.eval(f"fetch({json.dumps(power_post)}, {{method: 'POST', credentials: 'same-origin'}}).then(r => r.status)")
                    await asyncio.sleep(1.5)
                del_status = await page.eval(f"""(async () => {{
                    const fd = new FormData();
                    fd.append('name', {json.dumps(inst_name)});
                    const r = await fetch('/delete_instance?inst_id={inst_id}', {{method: 'POST', body: fd, credentials: 'same-origin'}});
                    return r.status;
                }})()""")
                if not del_status or int(del_status) >= 400:
                    raise RuntimeError(f"Failed to delete Solveit instance {inst_name!r} (id={inst_id}, HTTP {del_status}).")
                await asyncio.sleep(1.0)
                if self._match_instance(await self._goto_dashboard(page), inst_name, inst_id=inst_id) is not None:
                    raise RuntimeError(f"Solveit instance {inst_name!r} (id={inst_id}) still present after delete request.")
                removed_live = True
        finally: await self._close_session(td, cdp)

    if existed_in_state:
        state.pop(review_app_id, None)
        self._save_state(state)
    return removed_live or existed_in_state

In [ ]:
show_doc(FastCDPReviewAppProvisioner)

Offline verification of cookie filtering and dashboard instance card matching.

In [ ]:
with tempfile.TemporaryDirectory() as td:
    auth_path = Path(td)/"solveit-state.json"
    auth_path.write_json(dict(cookies=[dict(name='session', value='abc', domain='.solve.it.com', path='/', expires=1999999999),
        dict(name='', value='ignored')]))
    cookies = _load_cdp_cookies(auth_path)
    test_eq((len(cookies), cookies[0]["name"], cookies[0]["expires"]), (1, "session", 1999999999.0))

cards = [dict(inst_id="99", name="review-foo"), dict(inst_id="100", name="review-bar")]
test_eq(FastCDPReviewAppProvisioner._match_instance(cards, "review-bar")["inst_id"], "100")
test_eq(FastCDPReviewAppProvisioner._match_instance(cards, "other", inst_id="99")["name"], "review-foo")
assert FastCDPReviewAppProvisioner._match_instance(cards, "missing") is None

### Live Solveit Dashboard Inspection Example

Tagged `#| live` so `nbdev-test` skips it during offline runs.

In [ ]:
#| live
async def _live_solveit_check():
    prov = FastCDPReviewAppProvisioner()
    td, cdp, page = await prov._open_session()
    try:
        instances = await prov._goto_dashboard(page)
        print(f"Visible Solveit instances: {len(instances)}")
    finally: await prov._close_session(td, cdp)

await _live_solveit_check()